# Forest Cover Type Classification with Spark MLlib

**Big Data Analytics and Text Mining — Module 2**
Luca Mongiello & Xiaojun Liang — University of Bologna

Supervised multiclass classification of forest cover type (7 classes) on the
UCI **Covertype** dataset (N = 581,012, p = 54), using Spark MLlib on a 3-node
cluster (1 master + 2 workers), with data on **HDFS** and Spark running on **YARN**.

---

## Environment and required library versions

This notebook is meant to run on the cluster provisioned by the project's
`Vagrantfile` + `bootstrap.sh`. To run it as-is, the following versions are required:

| Component   | Version |
|-------------|---------|
| Spark       | 4.1.2   |
| Hadoop      | 3.5.0   |
| Java        | 17      |
| Python      | 3.12.3  |
| pyspark     | 4.1.2   |
| pandas      | 3.0.3   |
| numpy       | 2.5.0   |
| matplotlib  | 3.11.0  |
| seaborn     | 0.13.2  |

The cell below prints the versions actually present at runtime, so any mismatch
with the table above is immediately visible.

In [1]:
import sys
import pyspark, pandas, numpy, matplotlib, seaborn

print("Python    ", sys.version.split()[0])
print("pyspark   ", pyspark.__version__)
print("pandas    ", pandas.__version__)
print("numpy     ", numpy.__version__)
print("matplotlib", matplotlib.__version__)
print("seaborn   ", seaborn.__version__)

Python     3.12.3
pyspark    4.1.2
pandas     3.0.3
numpy      2.5.0
matplotlib 3.11.0
seaborn    0.13.2


## 1. Configuration

All environment-specific values live here, in a single block. Nothing
downstream hard-codes paths, resources, or seeds: to adapt the notebook to a
different cluster, only this cell needs to change.

In [2]:
class CONFIG:
    # --- Spark on YARN ---
    SPARK_MASTER       = "yarn"
    DEPLOY_MODE        = "client"
    APP_NAME           = "covertype"
    DRIVER_MEMORY      = "1g"        
    EXECUTOR_INSTANCES = 1           
    EXECUTOR_CORES     = 3
    EXECUTOR_MEMORY    = "2500m"
    EXECUTOR_OVERHEAD  = "512m"   
    AM_MEMORY          = "512m"      
    SHUFFLE_PARTITIONS = 12          

    # --- Data on HDFS ---
    HDFS_DIR  = "hdfs:///user/vagrant/covtype"
    HDFS_RAW  = HDFS_DIR + "/covtype.data.gz"
    LOCAL_RAW = "/home/vagrant/covtype.data.gz"   # staging path on the driver
    DATA_URL  = ("https://archive.ics.uci.edu/ml/"
                 "machine-learning-databases/covtype/covtype.data.gz")

    # --- Reproducibility / splits ---
    SEED       = 42
    TRAIN_FRAC = 0.8
    TEST_FRAC  = 0.2

## 2. SparkSession on YARN

The session is built explicitly from `CONFIG` (we launch a plain Jupyter, not the
`pyspark` wrapper, precisely so that we control resources here). After creation
the cell prints the Spark version, the application id, and the YARN tracking URL,
where the running application should appear under
[192.168.56.10:8088](http://192.168.56.10:8088).

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName(CONFIG.APP_NAME)
    .master(CONFIG.SPARK_MASTER)
    .config("spark.submit.deployMode",      CONFIG.DEPLOY_MODE)
    .config("spark.executor.instances",     CONFIG.EXECUTOR_INSTANCES)
    .config("spark.executor.cores",         CONFIG.EXECUTOR_CORES)
    .config("spark.executor.memory",        CONFIG.EXECUTOR_MEMORY)
    .config("spark.executor.memoryOverhead",CONFIG.EXECUTOR_OVERHEAD)
    .config("spark.yarn.am.memory",         CONFIG.AM_MEMORY)
    .config("spark.driver.memory",          CONFIG.DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", CONFIG.SHUFFLE_PARTITIONS)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Application ID:", spark.sparkContext.applicationId)
print("Master        :", spark.sparkContext.master)
print("Tracking UI   : http://192.168.56.10:8088")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 11:35:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/24 11:35:39 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark version : 4.1.2
Application ID: application_1782300550185_0001
Master        : yarn
Tracking UI   : http://192.168.56.10:8088


## 3. Data acquisition on HDFS

The dataset is downloaded once to the driver, then copied to HDFS under
`/user/vagrant/covtype/`. The step is **idempotent**: if the file already exists
on HDFS it is not re-downloaded or re-uploaded, so re-running the notebook is safe.

Note: `covtype.data.gz` is gzip-compressed and therefore not splittable, so Spark
will read it with a single task — fine for a ~11 MB file. We will repartition right
after loading (next section) so the 581k rows are distributed across the executors.

In [4]:
import os, subprocess, urllib.request

os.environ["HADOOP_ROOT_LOGGER"] = "ERROR,console"

def hdfs_exists(path: str) -> bool:
    return subprocess.run(["hdfs", "dfs", "-test", "-e", path]).returncode == 0

def sh(cmd: list):
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)

# 1. Ensure the HDFS target directory exists
sh(["hdfs", "dfs", "-mkdir", "-p", CONFIG.HDFS_DIR])

# 2. Download + put only if the file is not already on HDFS
if hdfs_exists(CONFIG.HDFS_RAW):
    print("Already on HDFS:", CONFIG.HDFS_RAW)
else:
    if not os.path.exists(CONFIG.LOCAL_RAW):
        print("Downloading:", CONFIG.DATA_URL)
        urllib.request.urlretrieve(CONFIG.DATA_URL, CONFIG.LOCAL_RAW)
        print("Saved to:", CONFIG.LOCAL_RAW)
    sh(["hdfs", "dfs", "-put", "-f", CONFIG.LOCAL_RAW, CONFIG.HDFS_RAW])
    print("Uploaded to HDFS:", CONFIG.HDFS_RAW)

# 3. Confirm
sh(["hdfs", "dfs", "-ls", CONFIG.HDFS_DIR])

+ hdfs dfs -mkdir -p hdfs:///user/vagrant/covtype
Downloading: https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz
Saved to: /home/vagrant/covtype.data.gz
+ hdfs dfs -put -f /home/vagrant/covtype.data.gz hdfs:///user/vagrant/covtype/covtype.data.gz
Uploaded to HDFS: hdfs:///user/vagrant/covtype/covtype.data.gz
+ hdfs dfs -ls hdfs:///user/vagrant/covtype
Found 1 items
-rw-r--r--   1 vagrant supergroup   11240707 2026-06-24 11:36 hdfs:///user/vagrant/covtype/covtype.data.gz


## 4. Schema definition and data loading

The schema is declared **explicitly** (no inference): every one of the 55 columns
of `covtype.data` is an integer — 10 quantitative features, 4 wilderness-area
indicators, 40 soil-type indicators, and the `Cover_Type` label (values 1..7).

After loading from HDFS we `repartition` (the gzip file is read by a single task,
so the raw DataFrame has only one partition) and `cache` the result, because the
exploratory analysis will reuse it many times. The final `count()` is the first
real Spark *action* and triggers the actual distributed read on the cluster.

Note: `Cover_Type` ranges 1..7 here; MLlib expects labels starting at 0, so we will
remap it during preprocessing — not now, to keep the load faithful to the raw data.

In [5]:
from pyspark.sql.types import StructType, StructField, IntegerType

# --- Column groups (explicit, reused downstream) ---
QUANTITATIVE = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]                                                          # 10 quantitative
WILDERNESS = [f"Wilderness_Area_{i}" for i in range(4)]    # 4 binary
SOIL       = [f"Soil_Type_{i}"      for i in range(40)]    # 40 binary
LABEL      = "Cover_Type"                                  # target (1..7)

FEATURE_COLS = QUANTITATIVE + WILDERNESS + SOIL           # 54 features
ALL_COLS     = FEATURE_COLS + [LABEL]                     # 55 columns

# Every column is an integer -> a single explicit schema, no inference.
schema = StructType([
    StructField(name, IntegerType(), nullable=False) for name in ALL_COLS
])

print(f"Schema defined: {len(schema.fields)} columns "
      f"({len(QUANTITATIVE)} quantitative + {len(WILDERNESS)} wilderness "
      f"+ {len(SOIL)} soil + 1 label).")

Schema defined: 55 columns (10 quantitative + 4 wilderness + 40 soil + 1 label).


In [6]:
# --- Load the raw .gz from HDFS using the explicit schema ---
df_raw = (
    spark.read
    .schema(schema)             # no inferSchema: types are fixed above
    .option("header", "false")  # covtype.data has no header row
    .csv(CONFIG.HDFS_RAW)
)

# gzip is not splittable -> df_raw has 1 partition. Repartition to spread the
# 581k rows across executors, then cache (the EDA reuses this DataFrame a lot).
df = df_raw.repartition(CONFIG.SHUFFLE_PARTITIONS).cache()

# First real ACTION: triggers read + repartition + cache on the cluster.
n_rows = df.count()

print(f"Rows loaded : {n_rows:,}")
print(f"Partitions  : {df.rdd.getNumPartitions()}")
print(f"Columns     : {len(df.columns)}")
print(f"Matches expected 581,012 rows: {n_rows == 581012}")

26/06/24 11:36:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Rows loaded : 581,012
Partitions  : 12
Columns     : 55
Matches expected 581,012 rows: True


In [7]:
df.printSchema()
df.select(QUANTITATIVE + [LABEL]).limit(5).toPandas()

root
 |-- Elevation: integer (nullable = true)
 |-- Aspect: integer (nullable = true)
 |-- Slope: integer (nullable = true)
 |-- Horizontal_Distance_To_Hydrology: integer (nullable = true)
 |-- Vertical_Distance_To_Hydrology: integer (nullable = true)
 |-- Horizontal_Distance_To_Roadways: integer (nullable = true)
 |-- Hillshade_9am: integer (nullable = true)
 |-- Hillshade_Noon: integer (nullable = true)
 |-- Hillshade_3pm: integer (nullable = true)
 |-- Horizontal_Distance_To_Fire_Points: integer (nullable = true)
 |-- Wilderness_Area_0: integer (nullable = true)
 |-- Wilderness_Area_1: integer (nullable = true)
 |-- Wilderness_Area_2: integer (nullable = true)
 |-- Wilderness_Area_3: integer (nullable = true)
 |-- Soil_Type_0: integer (nullable = true)
 |-- Soil_Type_1: integer (nullable = true)
 |-- Soil_Type_2: integer (nullable = true)
 |-- Soil_Type_3: integer (nullable = true)
 |-- Soil_Type_4: integer (nullable = true)
 |-- Soil_Type_5: integer (nullable = true)
 |-- Soil_Type

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Cover_Type
0,2773,44,14,90,21,1409,222,208,117,2219,2
1,3235,179,10,450,130,450,225,246,154,631,1
2,2894,89,11,124,21,3303,236,223,117,4696,2
3,3005,352,13,153,25,4290,196,217,158,854,2
4,3285,326,15,524,147,3818,182,222,179,2106,1
